# Semantic Search: a small demo

Keyword search matches **words**. Semantic search matches **meaning**.
Steps: embed the documents -> embed the query -> cosine similarity -> rank.

No database here. A numpy array is our "vector store".

Install once: `pip install sentence-transformers numpy`

## 1. Our small collection of documents

In [ ]:
docs = [
    "The cat sat on the mat",
    "A puppy is playing in the garden",
    "Kittens love to chase string and sleep in the sun",
    "How to cook spaghetti with tomato sauce",
    "The best pizza recipe with fresh cheese",
    "A new electric car with a long battery range",
    "Stock markets fell sharply on Monday",
    "The football team won the championship final",
]
for i, d in enumerate(docs):
    print(i, d)

## 2. Load an embedding model
`all-MiniLM-L6-v2` is small (about 90 MB), runs on a normal laptop CPU, and turns any sentence into **384 numbers**.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

## 3. Embed the documents (done once)
Each document becomes one vector. `normalize_embeddings=True` makes every vector length 1, which makes cosine similarity just a dot product.

In [ ]:
doc_vectors = model.encode(docs, normalize_embeddings=True)

print("shape:", doc_vectors.shape, " -> 8 documents, 384 numbers each")
print("first 8 numbers of doc 0:", doc_vectors[0][:8].round(3))

## 4. Cosine similarity by hand
cosine(A, B) = (A . B) / (|A| x |B|). With length-1 vectors the bottom is 1, so it is just `A . B`.
Closer to 1 means more similar meaning.

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = model.encode("cat")
for word in ["kitten", "dog", "pizza", "car"]:
    print(f"cat vs {word:<7} {cosine(a, model.encode(word)):.3f}")

## 5. Search function
Embed the query with the **same model**, compare to every document vector, sort, return the top results.

In [ ]:
def search(query, k=3):
    q = model.encode(query, normalize_embeddings=True)
    scores = doc_vectors @ q               # cosine similarity with every document
    top = np.argsort(scores)[::-1][:k]     # indices of the highest scores
    for i in top:
        print(f"{scores[i]:.3f}  {docs[i]}")

search("feline pets")

Notice: no document contains the word *feline*, yet the cat and kitten documents come first.

In [ ]:
search("something to eat for dinner")
print()
search("vehicle that runs on electricity")
print()
search("sports victory")

## 6. Compare with keyword search
A simple word-overlap search: count how many query words appear in each document.

In [ ]:
def keyword_search(query, k=3):
    q_words = set(query.lower().split())
    scored = [(len(q_words & set(d.lower().split())), d) for d in docs]
    scored.sort(reverse=True)
    for score, d in scored[:k]:
        print(f"{score}  {d}")

query = "feline pets"
print("KEYWORD:"); keyword_search(query)
print("\nSEMANTIC:"); search(query)

## 7. Try it yourself
1. Add 5 documents of your own to `docs` (remember to re-run the embedding cell).
2. Write a query that shares **no words** with the best document. Does semantic search still find it?
3. Find a query where semantic search gets it wrong. Why might that happen?
4. Search an exact code like `"ERROR 404"`. Which method is better for that?

**Next:** with millions of documents, comparing against every vector is too slow. A *vector database* (FAISS, Chroma, pgvector) stores the vectors and finds neighbours fast. The idea is the same as `doc_vectors @ q`.